In [ ]:

pkgs <- c("data.table", "ggplot2", "foreach", "doParallel", "microbenchmark",
          "purrr", "arrow", "bench", "scales")
to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(to_install)) install.packages(to_install, repos = "https://cloud.r-project.org")
invisible(lapply(pkgs, library, character.only = TRUE))

YEAR          <- 2023
MONTHS        <- 1:6
DATA_DIR      <- "nyc_taxi_data"
PLOT_DIR      <- "plots"
USE_SYNTHETIC <- FALSE
N_SYNTH       <- 500000
set.seed(123)
options(timeout = 1200, datatable.print.nrows = 20)
dir.create(DATA_DIR, showWarnings = FALSE); dir.create(PLOT_DIR, showWarnings = FALSE)

tm <- function(expr) unname(system.time(expr)["elapsed"])

cat("\n==== TASK 1: DATA ACQUISITION & PREPARATION ====\n")

base_url <- "https://d37ci6vzurychx.cloudfront.net"
zone_file <- file.path(DATA_DIR, "taxi_zone_lookup.csv")
if (!USE_SYNTHETIC && !file.exists(zone_file))
  download.file(paste0(base_url, "/misc/taxi_zone_lookup.csv"), zone_file, mode = "wb")

keep_cols <- c("tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
               "trip_distance", "PULocationID", "DOLocationID", "payment_type",
               "fare_amount", "tip_amount", "tolls_amount", "total_amount")

gen_synth <- function(m, n = N_SYNTH) {
  start <- as.POSIXct(sprintf("%d-%02d-01", YEAR, m), tz = "UTC")
  pu <- start + runif(n, 0, 27 * 86400)
  dist <- round(rlnorm(n, 0.7, 0.8), 2)
  dur  <- pmax(60, dist * 240 + rnorm(n, 300, 120))
  fare <- round(3 + 2.9 * dist + rnorm(n, 0, 1.5), 2)
  tip  <- round(pmax(0, fare * runif(n, 0, 0.3)), 2)
  data.table(tpep_pickup_datetime = pu, tpep_dropoff_datetime = pu + dur,
             passenger_count = sample(1:4, n, TRUE, c(.7, .15, .1, .05)),
             trip_distance = dist, PULocationID = sample(1:263, n, TRUE),
             DOLocationID = sample(1:263, n, TRUE),
             payment_type = sample(1:4, n, TRUE, c(.75, .22, .02, .01)),
             fare_amount = fare, tip_amount = tip, tolls_amount = 0,
             total_amount = fare + tip + 0.5 + 2.5)
}

read_month <- function(m) {
  if (USE_SYNTHETIC) { dt <- gen_synth(m) } else {
    f <- file.path(DATA_DIR, sprintf("yellow_tripdata_%d-%02d.parquet", YEAR, m))
    if (!file.exists(f))
      download.file(sprintf("%s/trip-data/yellow_tripdata_%d-%02d.parquet", base_url, YEAR, m),
                    f, mode = "wb")
    dt <- as.data.table(arrow::read_parquet(f, col_select = all_of(keep_cols)))
  }
  dt[, src_month := m]
  dt
}

t_load <- tm({ dt <- rbindlist(lapply(MONTHS, read_month), use.names = TRUE) })
cat(sprintf("Loaded %s rows x %d cols in %.1f s\n", comma(nrow(dt)), ncol(dt), t_load))

cat("\nDimensions:", dim(dt), "\n"); str(dt); print(summary(dt))
cat("\nMissing values per column:\n"); print(colSums(is.na(dt)))

n_raw  <- nrow(dt)
n_dups <- n_raw - uniqueN(dt)
cat(sprintf("Duplicate rows: %s\n", comma(n_dups)))
dt <- unique(dt)

dt[, duration_min := as.numeric(difftime(tpep_dropoff_datetime, tpep_pickup_datetime, units = "mins"))]
cat("\nInvalid-record counts BEFORE cleaning:\n")
print(dt[, .(neg_or_zero_fare = sum(fare_amount <= 0, na.rm = TRUE),
             zero_distance    = sum(trip_distance <= 0, na.rm = TRUE),
             distance_gt_100  = sum(trip_distance > 100, na.rm = TRUE),
             bad_passengers   = sum(!passenger_count %in% 1:6, na.rm = TRUE),
             bad_duration     = sum(duration_min <= 0 | duration_min > 360, na.rm = TRUE),
             wrong_month      = sum(month(tpep_pickup_datetime) != src_month |
                                    year(tpep_pickup_datetime) != YEAR, na.rm = TRUE))])

dt <- dt[!is.na(passenger_count) & passenger_count %in% 1:6 &
         fare_amount > 0 & total_amount > 0 &
         trip_distance > 0 & trip_distance < 100 &
         duration_min > 0 & duration_min <= 360 &
         PULocationID %between% c(1, 265) & DOLocationID %between% c(1, 265) &
         month(tpep_pickup_datetime) == src_month & year(tpep_pickup_datetime) == YEAR]
cat(sprintf("Rows kept after cleaning: %s of %s (%.1f%%)\n",
            comma(nrow(dt)), comma(n_raw), 100 * nrow(dt) / n_raw))

mem_before <- as.numeric(object.size(dt)) / 1024^2
pay_labels <- c("Flex Fare", "Credit card", "Cash", "No charge", "Dispute", "Unknown")
dt[, `:=`(hour        = hour(tpep_pickup_datetime),
          day         = mday(tpep_pickup_datetime),
          day_of_week = factor(wday(tpep_pickup_datetime), levels = 1:7,
                               labels = c("Sun","Mon","Tue","Wed","Thu","Fri","Sat")),
          month       = month(tpep_pickup_datetime),
          passenger_count = as.integer(passenger_count),
          PULocationID    = as.integer(PULocationID),
          DOLocationID    = as.integer(DOLocationID),
          payment_type    = factor(payment_type, levels = 0:5, labels = pay_labels))]
dt[, speed_mph := trip_distance / (duration_min / 60)]
dt <- dt[speed_mph < 80]


dt[, c("tpep_pickup_datetime", "tpep_dropoff_datetime", "src_month") := NULL]
dt[, hour := as.integer(hour)]
invisible(gc())
mem_after <- as.numeric(object.size(dt)) / 1024^2
cat(sprintf("Memory: %.0f MB before type-conversion/column-drop -> %.0f MB after\n",
            mem_before, mem_after))


zones <- fread(zone_file)
setnames(zones, tolower(names(zones)))
setkey(zones, locationid)
dt[zones, on = .(PULocationID = locationid), `:=`(PU_Borough = i.borough, PU_Zone = i.zone)]
dt[zones, on = .(DOLocationID = locationid), `:=`(DO_Borough = i.borough, DO_Zone = i.zone)]
dt[is.na(PU_Zone),  `:=`(PU_Zone = "Unknown", PU_Borough = "Unknown")]
dt[is.na(DO_Zone),  `:=`(DO_Zone = "Unknown", DO_Borough = "Unknown")]
print(head(dt))
cat("Clean dataset:", comma(nrow(dt)), "rows\n")

cat("\n==== TASK 2: TRANSPORTATION DATA ANALYTICS ====\n")
month_lab <- function(x) month.abb[x]

trips_hour <- dt[, .(trips = .N, avg_fare = mean(fare_amount),
                     revenue = sum(total_amount)), keyby = hour]
trips_dow  <- dt[, .(trips = .N, avg_fare = mean(fare_amount),
                     revenue = sum(total_amount)), keyby = day_of_week]
trips_mon  <- dt[, .(trips = .N, avg_fare = mean(fare_amount),
                     revenue = sum(total_amount)), keyby = month]
cat("\n(1) Trips by hour:\n");        print(trips_hour)
cat("\n(2) Trips by day of week:\n"); print(trips_dow)
cat("\n(3,4) Monthly usage / fare / revenue:\n"); print(trips_mon)
cat(sprintf("\nPeak hour: %02d:00 (%s trips) | Busiest day: %s\n",
            trips_hour[which.max(trips), hour], comma(max(trips_hour$trips)),
            trips_dow[which.max(trips), day_of_week]))

routes <- dt[, .(trips = .N, revenue = sum(total_amount), avg_fare = mean(total_amount),
                 avg_dist = mean(trip_distance)),
             by = .(PULocationID, DOLocationID, PU_Zone, DO_Zone)]
routes[, route := paste(PU_Zone, "->", DO_Zone)]
top_routes_freq <- routes[order(-trips)][1:10]
top_routes_rev  <- routes[order(-revenue)][1:10]
cat("\n(5) Most frequent routes:\n");  print(top_routes_freq[, .(route, trips, avg_fare)])
cat("\n(6) Highest-revenue routes:\n"); print(top_routes_rev[, .(route, trips, revenue)])

top_pu <- dt[, .(trips = .N), by = PU_Zone][order(-trips)][1:10]
top_do <- dt[, .(trips = .N), by = DO_Zone][order(-trips)][1:10]

cat(sprintf("\n(7) Pearson correlation distance vs fare: %.4f\n",
            dt[, cor(trip_distance, fare_amount)]))
samp <- dt[sample(.N, min(.N, 50000))]
fit  <- lm(fare_amount ~ trip_distance, data = samp)
print(summary(fit)$coefficients)
cat(sprintf("Interpretation: base fare ~ $%.2f + $%.2f per mile (R^2 = %.3f)\n",
            coef(fit)[1], coef(fit)[2], summary(fit)$r.squared))


pay_borough <- dt[PU_Borough != "Unknown", .N, by = .(PU_Borough, payment_type)]
pay_borough[, share := N / sum(N), by = PU_Borough]
cat("\n(8) Payment type share by pickup borough:\n")
print(dcast(pay_borough, PU_Borough ~ payment_type, value.var = "share", fill = 0))
pay_overall <- dt[, .N, by = payment_type][, pct := round(100 * N / sum(N), 2)][order(-N)]


tip_pay <- dt[payment_type %in% c("Credit card", "Cash"),
              .(avg_tip_pct = 100 * sum(tip_amount) / sum(fare_amount)), by = payment_type]
speed_hour <- dt[, .(avg_speed_mph = mean(speed_mph)), keyby = hour]
airport <- dt[grepl("Airport", PU_Zone) | grepl("Airport", DO_Zone),
              .(trips = .N, avg_fare = mean(fare_amount), avg_total = mean(total_amount))]
cat("\n(9) Tip % by payment type (cash tips are not recorded):\n"); print(tip_pay)
cat("\nAverage speed by hour (mph):\n"); print(speed_hour)
cat("\nAirport trips:\n"); print(airport)

cat("\n==== TASK 3: FUNCTIONAL vs VECTORIZED vs DATA.TABLE ====\n")

hours <- 0:23
fare_v <- dt$fare_amount; hour_v <- dt$hour
df_base <- data.frame(fare_amount = fare_v, hour = hour_v)

opA_sapply   <- function() sapply(hours, function(h) mean(fare_v[hour_v == h]))
opA_lapply   <- function() unlist(lapply(hours, function(h) mean(fare_v[hour_v == h])))
opA_purrr    <- function() purrr::map_dbl(hours, ~ mean(fare_v[hour_v == .x]))
opA_tapply   <- function() as.numeric(tapply(fare_v, hour_v, mean))
opA_aggr     <- function() aggregate(fare_amount ~ hour, data = df_base, FUN = mean)$fare_amount
opA_vector   <- function() as.numeric(rowsum(fare_v, hour_v)) / tabulate(hour_v + 1L)
opA_dt       <- function() dt[, mean(fare_amount), keyby = hour]$V1

ref <- opA_dt()
stopifnot(all.equal(ref, opA_sapply()), all.equal(ref, opA_purrr()),
          all.equal(ref, opA_vector()), all.equal(ref, opA_aggr()))
cat("All Operation-A implementations give identical results.\n")

mbA <- microbenchmark(
  `Functional: sapply`   = opA_sapply(),
  `Functional: lapply`   = opA_lapply(),
  `Functional: purrr`    = opA_purrr(),
  `Base R: tapply`       = opA_tapply(),
  `Base R: aggregate`    = opA_aggr(),
  `Vectorized: rowsum`   = opA_vector(),
  `data.table`           = opA_dt(),
  times = 10, unit = "ms")
print(mbA)
resA <- as.data.table(summary(mbA, unit = "ms"))[, .(Operation = "A: mean fare by hour (full data)",
                                                    Approach = as.character(expr), Median_ms = median)]

N_ROW <- 200000
s   <- dt[sample(.N, min(.N, N_ROW)), .(fare_amount, trip_distance)]
mat <- as.matrix(s)
opB_apply  <- function() apply(mat, 1, function(r) r[1] / r[2])
opB_map2   <- function() purrr::map2_dbl(s$fare_amount, s$trip_distance, ~ .x / .y)
opB_mapply <- function() mapply(function(f, d) f / d, s$fare_amount, s$trip_distance)
opB_vector <- function() mat[, 1] / mat[, 2]
opB_dt     <- function() s[, fare_amount / trip_distance]
stopifnot(all.equal(unname(opB_apply()), opB_vector()),
          all.equal(unname(opB_map2()), opB_vector()))

mbB <- microbenchmark(
  `Functional: apply`  = opB_apply(),
  `Functional: mapply` = opB_mapply(),
  `Functional: purrr::map2` = opB_map2(),
  `Vectorized`         = opB_vector(),
  `data.table`         = opB_dt(),
  times = 5, unit = "ms")
print(mbB)
resB <- as.data.table(summary(mbB, unit = "ms"))[, .(Operation = sprintf("B: fare/mile row-wise (%s rows)", comma(nrow(s))),
                                                    Approach = as.character(expr), Median_ms = median)]
memA <- bench::mark(sapply = opA_sapply(), purrr = opA_purrr(), aggregate = opA_aggr(),
                    vectorized = opA_vector(), data.table = opA_dt(),
                    iterations = 5, check = FALSE)
memB <- bench::mark(apply = opB_apply(), purrr_map2 = opB_map2(),
                    vectorized = opB_vector(), data.table = opB_dt(),
                    iterations = 3, check = FALSE)
cat("\nMemory allocated - Operation A:\n");
print(data.table(approach = as.character(memA$expression), mem_alloc = format(memA$mem_alloc)))
cat("\nMemory allocated - Operation B:\n");
print(data.table(approach = as.character(memB$expression), mem_alloc = format(memB$mem_alloc)))

opC_base <- function() aggregate(total_amount ~ hour + PU_Borough + payment_type,
                                 data = dt_sub_df, FUN = sum)
dt_sub    <- dt[, .(total_amount, hour, PU_Borough, payment_type)]
dt_sub_df <- as.data.frame(dt_sub)
opC_dt    <- function() dt_sub[, .(total = sum(total_amount)), by = .(hour, PU_Borough, payment_type)]
mbC <- microbenchmark(`Base R: aggregate` = opC_base(), `data.table` = opC_dt(),
                      times = 3, unit = "s")
print(mbC)
resC <- as.data.table(summary(mbC, unit = "ms"))[, .(Operation = "C: group-by hour x borough x payment (full data)",
                                                    Approach = as.character(expr), Median_ms = median)]


cat("\n==== TASK 4: SEQUENTIAL vs PARALLEL ====\n")


month_list <- split(dt[, .(month, trip_distance, fare_amount)], by = "month", keep.by = TRUE)
cat("Partitions:", names(month_list), "| rows:", sapply(month_list, nrow), "\n")


heavy_task <- function(d, B = 200, n_samp = 100000) {
  set.seed(d$month[1])
  x <- d$trip_distance; y <- d$fare_amount; n <- length(x)
  m <- min(n_samp, n)
  slopes <- numeric(B)
  for (b in seq_len(B)) {
    idx <- sample.int(n, m, replace = TRUE)
    slopes[b] <- lm.fit(cbind(1, x[idx]), y[idx])$coefficients[2]
  }
  data.table(month = d$month[1], slope_mean = mean(slopes),
             ci_low = quantile(slopes, 0.025), ci_high = quantile(slopes, 0.975))
}
light_task <- function(d) d[, .(month = month[1], avg_fare = mean(fare_amount))]


t_seq <- tm({ res_seq <- rbindlist(lapply(month_list, heavy_task)) })
cat(sprintf("Sequential (heavy task): %.2f s\n", t_seq)); print(res_seq)


max_cores  <- max(1, parallel::detectCores() - 1, na.rm = TRUE)
core_set   <- sort(unique(pmin(c(2, 4, max_cores), length(month_list), max_cores)))
par_results <- data.table()

for (nc in core_set) {
  cl <- makeCluster(nc); registerDoParallel(cl)
  t_par <- tm({
    res_par <- foreach(d = month_list, .combine = rbind, .packages = "data.table",
                       .export = "heavy_task") %dopar% heavy_task(d)
  })
  stopCluster(cl)
  par_results <- rbind(par_results,
                       data.table(cores = nc, par_time_s = t_par,
                                  speedup = t_seq / t_par, efficiency_pct = 100 * t_seq / t_par / nc))
}
cat("\nSpeedup = Sequential time / Parallel time\n"); print(par_results)
stopifnot(isTRUE(all.equal(res_seq$slope_mean, res_par$slope_mean)))

t_seq_l <- tm(rbindlist(lapply(month_list, light_task)))
cl <- makeCluster(max(core_set)); registerDoParallel(cl)
t_par_l <- tm(foreach(d = month_list, .combine = rbind, .packages = "data.table",
                      .export = "light_task") %dopar% light_task(d))
stopCluster(cl)
cat(sprintf("\nLight task: sequential %.3f s | parallel %.3f s | speedup %.2fx (<1 = parallel slower)\n",
            t_seq_l, t_par_l, t_seq_l / t_par_l))

best_par <- par_results[which.max(speedup)]

cat("\n==== TASK 5: BENCHMARK COMPARISON TABLE ====\n")
resD <- data.table(
  Operation = c("D: bootstrap, 6 monthly partitions (heavy)", "D: bootstrap, 6 monthly partitions (heavy)",
                "E: mean fare per month (light)", "E: mean fare per month (light)"),
  Approach  = c("Sequential (lapply)", sprintf("Parallel foreach/doParallel (%d cores)", best_par$cores),
                "Sequential (lapply)", sprintf("Parallel foreach/doParallel (%d cores)", max(core_set))),
  Median_ms = 1000 * c(t_seq, best_par$par_time_s, t_seq_l, t_par_l))
bench_table <- rbindlist(list(resA, resB, resC, resD))
bench_table[, Relative_to_fastest := round(Median_ms / min(Median_ms), 1), by = Operation]
bench_table[, Median_ms := round(Median_ms, 2)]
print(bench_table[order(Operation, Median_ms)], nrows = 100)
fwrite(bench_table, "benchmark_comparison_table.csv")

cat("\nFastest approach per operation:\n")
print(bench_table[, .SD[which.min(Median_ms)], by = Operation][, .(Operation, Approach, Median_ms)])

cat("\n==== TASK 6: VISUALIZATIONS (saved in ./plots) ====\n")
th <- theme_minimal(base_size = 12) + theme(plot.title = element_text(face = "bold"))
save_p <- function(p, name) { print(p); ggsave(file.path(PLOT_DIR, name), p, width = 9, height = 5, dpi = 150) }


save_p(ggplot(trips_hour, aes(hour, trips)) + geom_col(fill = "#2c7fb8") +
  scale_x_continuous(breaks = 0:23) + scale_y_continuous(labels = comma) +
  labs(title = "Taxi Trips by Hour of Day", subtitle = "Evening rush (17-19h) is the demand peak; lowest at 3-5 am",
       x = "Hour of day (0-23)", y = "Number of trips") + th, "01_trips_by_hour.png")

save_p(ggplot(trips_dow, aes(day_of_week, trips, fill = day_of_week)) + geom_col(show.legend = FALSE) +
  scale_y_continuous(labels = comma) +
  labs(title = "Taxi Demand by Day of Week", subtitle = "Weekdays (esp. Thu/Fri) typically busier than Sunday",
       x = "Day of week", y = "Number of trips") + th, "02_trips_by_dow.png")

save_p(ggplot(trips_mon, aes(month, trips)) + geom_line(colour = "#d95f0e", linewidth = 1.2) + geom_point(size = 3) +
  scale_x_continuous(breaks = trips_mon$month, labels = month_lab) + scale_y_continuous(labels = comma) +
  labs(title = "Monthly Taxi Demand", x = "Month", y = "Number of trips") + th, "03_monthly_demand.png")


save_p(ggplot(trips_hour, aes(hour, avg_fare)) + geom_line(colour = "#31a354", linewidth = 1.2) + geom_point() +
  scale_x_continuous(breaks = 0:23) +
  labs(title = "Average Fare by Hour", subtitle = "Early-morning fares are higher (airport/long trips)",
       x = "Hour of day", y = "Average fare (USD)") + th, "04_avg_fare_hour.png")
save_p(ggplot(trips_mon, aes(month, revenue / 1e6)) + geom_col(fill = "#756bb1") +
  scale_x_continuous(breaks = trips_mon$month, labels = month_lab) +
  labs(title = "Monthly Revenue (total_amount)", x = "Month", y = "Revenue (million USD)") + th, "05_monthly_revenue.png")

save_p(ggplot(top_pu, aes(reorder(PU_Zone, trips), trips)) + geom_col(fill = "#3182bd") + coord_flip() +
  scale_y_continuous(labels = comma) + labs(title = "Top 10 Pickup Zones", x = "Pickup zone", y = "Trips") + th,
  "06_top_pickups.png")
save_p(ggplot(top_do, aes(reorder(DO_Zone, trips), trips)) + geom_col(fill = "#e6550d") + coord_flip() +
  scale_y_continuous(labels = comma) + labs(title = "Top 10 Drop-off Zones", x = "Drop-off zone", y = "Trips") + th,
  "07_top_dropoffs.png")


save_p(ggplot(top_routes_freq, aes(reorder(route, trips), trips)) + geom_col(fill = "#08519c") + coord_flip() +
  scale_y_continuous(labels = comma) + labs(title = "10 Most Frequent Routes", x = "Route (pickup -> drop-off)", y = "Trips") + th,
  "08_top_routes_freq.png")
save_p(ggplot(top_routes_rev, aes(reorder(route, revenue), revenue / 1e6)) + geom_col(fill = "#a50f15") + coord_flip() +
  labs(title = "10 Highest-Revenue Routes", x = "Route (pickup -> drop-off)", y = "Revenue (million USD)") + th,
  "09_top_routes_revenue.png")


save_p(ggplot(pay_overall, aes(reorder(payment_type, -N), N, fill = payment_type)) + geom_col(show.legend = FALSE) +
  scale_y_continuous(labels = comma) + labs(title = "Payment-Type Distribution", x = "Payment type", y = "Trips") + th,
  "10_payment_type.png")
save_p(ggplot(pay_borough, aes(PU_Borough, share, fill = payment_type)) + geom_col() +
  scale_y_continuous(labels = percent) +
  labs(title = "Payment Preference by Pickup Borough", x = "Pickup borough", y = "Share of trips", fill = "Payment type") +
  th + theme(axis.text.x = element_text(angle = 30, hjust = 1)), "11_payment_by_borough.png")

save_p(ggplot(samp[trip_distance < 40 & fare_amount < 200], aes(trip_distance, fare_amount)) +
  geom_point(alpha = 0.08, colour = "grey30") + geom_smooth(method = "lm", colour = "red") +
  labs(title = "Trip Distance vs Fare Amount", subtitle = sprintf("Fitted slope = $%.2f / mile", coef(fit)[2]),
       x = "Trip distance (miles)", y = "Fare amount (USD)") + th, "12_distance_vs_fare.png")

save_p(ggplot(bench_table, aes(reorder(Approach, -Median_ms), Median_ms, fill = Operation)) + geom_col(show.legend = FALSE) +
  facet_wrap(~ Operation, scales = "free", ncol = 2) + coord_flip() + scale_y_log10() +
  labs(title = "Benchmark: Median Execution Time (log scale)", x = NULL, y = "Median time (ms)") + th,
  "13_benchmark.png")
save_p(ggplot(par_results, aes(cores, speedup)) + geom_line(linewidth = 1.2, colour = "#e6550d") + geom_point(size = 3) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  labs(title = "Parallel Speedup vs Cores", subtitle = "Dashed = ideal linear speedup",
       x = "Number of cores", y = "Speedup (T_seq / T_par)") + th, "14_speedup.png")


cat("\n==== TASK 7: AUTO-GENERATED EVIDENCE FOR YOUR WRITE-UP ====\n")
fA <- resA[which.min(Median_ms)]; sA <- resA[Approach == "Functional: sapply"]
fB <- resB[which.min(Median_ms)]; sB <- resB[Approach == "Functional: apply"]
cat(sprintf("Op A fastest: %s (%.2f ms); sapply was %.1fx slower.\n", fA$Approach, fA$Median_ms, sA$Median_ms / fA$Median_ms))
cat(sprintf("Op B fastest: %s (%.2f ms); apply() was %.0fx slower.\n", fB$Approach, fB$Median_ms, sB$Median_ms / fB$Median_ms))
cat(sprintf("Op C: data.table was %.1fx faster than aggregate().\n",
            resC[Approach == "Base R: aggregate", Median_ms] / resC[Approach == "data.table", Median_ms]))
cat(sprintf("Parallel (heavy task): best speedup %.2fx with %d cores (efficiency %.0f%%).\n",
            best_par$speedup, best_par$cores, best_par$efficiency_pct))
cat(sprintf("Parallel (light task): speedup %.2fx -> parallel is not always faster (cluster start-up + data transfer overhead).\n",
            t_seq_l / t_par_l))
cat("\nNow write answers to the 8 questions of Task 7 in your notebook using the numbers above.\n")

sessionInfo()


==== TASK 1: DATA ACQUISITION & PREPARATION ====
Loaded 19,493,620 rows x 12 cols in 8.5 s

Dimensions: 19493620 12 
Classes ‘data.table’ and 'data.frame':	19493620 obs. of  12 variables:
 $ tpep_pickup_datetime : POSIXct, format: "2023-01-01 00:32:10" "2023-01-01 00:55:08" ...
 $ tpep_dropoff_datetime: POSIXct, format: "2023-01-01 00:40:36" "2023-01-01 01:01:27" ...
 $ passenger_count      : num  1 1 1 0 1 1 1 1 1 1 ...
 $ trip_distance        : num  0.97 1.1 2.51 1.9 1.43 1.84 1.66 11.7 2.95 3.01 ...
 $ PULocationID         : int  161 43 48 138 107 161 239 142 164 141 ...
 $ DOLocationID         : int  141 237 238 7 79 137 143 200 236 107 ...
 $ payment_type         : int  2 1 1 1 1 1 1 1 1 2 ...
 $ fare_amount          : num  9.3 7.9 14.9 12.1 11.4 12.8 12.1 45.7 17.7 14.9 ...
 $ tip_amount           : num  0 4 15 0 3.28 ...
 $ tolls_amount         : num  0 0 0 0 0 0 0 3 0 0 ...
 $ total_amount         : num  14.3 16.9 34.9 20.9 19.7 ...
 $ src_month            : int  1 1 1 1 1 1 1